# tabular_agent — Phase 3, step 8: aggregates that are looked up, and rows that are not replayed

Two tools land in this step, `fit_group_stats` and `filter_rows`. They look
unrelated and they arrived together for a reason: both are things a live run
reached for and could not have, and both sit on the same fault line — **what the
harness replays at prediction time, and what it does not.**

The design record is
[`0032`](../../../docs/decisions/0032-fitted-aggregates-and-a-step-that-is-not-replayed.md).
This notebook drives both by hand before letting an agent near them.

The general forms, stated before any code, because this is exactly where
dataset-specific thinking creeps into a library:

| the temptation, as a run phrases it | the capability that gets built |
|---|---|
| "average footfall for this store" | per-group statistics of a column, fitted once and replayed |
| "exclude the days the shop was shut" | some rows should not be trained on |

Neither docstring names a dataset. That is the test.

## 1. A panel to work on

Twelve stores over sixty days. Each store has a level of its own, one in five
store-days the shop is shut, and `footfall` is counted *on the day* — so it exists
in the training file and will not exist in a prediction request. That last part is
`0030`'s situation, and it is here on purpose: it is the case `fit_group_stats`
turns from a refusal into a usable feature.

In [ ]:
import os, shutil, json
import numpy as np, pandas as pd

WORK = os.path.abspath("_step8_demo")
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)

rng = np.random.default_rng(7)
n_stores, n_days = 12, 60
store = np.repeat([f"s{i:02d}" for i in range(n_stores)], n_days)
day = np.tile(np.arange(n_days), n_stores)

# Each store has a level and a footfall of its own — the thing a per-group
# statistic exists to capture.
level = pd.Series(store).map({f"s{i:02d}": 400 + 90 * i for i in range(n_stores)}).to_numpy()

# One store-day in five the shop is shut: no footfall, no sales.
is_open = (rng.random(len(store)) > 0.2).astype(int)

footfall = np.where(is_open, level / 5 + rng.normal(0, 6, len(store)), 0)
sales = np.where(is_open, level + 12 * footfall + rng.normal(0, 40, len(store)), 0)

df = pd.DataFrame({
    "store": store, "day": day, "is_open": is_open,
    "footfall": footfall.round(1), "sales": sales.round(1),
})
data_path = os.path.join(WORK, "panel.csv")
df.to_csv(data_path, index=False)
df.head()

In [ ]:
print("rows:             ", len(df))
print("closed store-days:", int((df["is_open"] == 0).sum()), f"({(df['is_open'] == 0).mean():.0%})")
print("corr(footfall, sales):",
      round(df.loc[df.is_open == 1, ["footfall", "sales"]].corr().iloc[0, 1], 3))

In [ ]:
from tabular_agent.harness import tools
from tabular_agent.harness.session import RunSession
from tabular_agent.taskspec import TaskSpec

session = RunSession(
    os.path.join(WORK, "run"),
    data_path,
    TaskSpec(target="sales", task_type="regression", metric="rmse", n_splits=4),
    # `footfall` is counted on the day being predicted. It is in the training
    # file and will not be in a prediction request (0030).
    serving_columns=["store", "day", "is_open"],
)
out = tools.inspect_data(session)
{k: out[k] for k in ("n_rows", "task_type", "metric", "columns_not_available_at_predict_time")}

## 2. What could not be built, and why the refusal was right

The obvious way to write "average footfall for this store" is one line of pandas.
`transform_features` refuses it. Read the message: as of this step, the last two
sentences are new — before, it said no such tool existed and told the agent not to
try again. That sentence sent a live run into a retry loop that killed it
(`0032`).

In [ ]:
from tabular_agent.sandbox import SandboxError

NAIVE = """
def transform_features(df):
    df = df.copy()
    df["store_footfall_mean"] = df.groupby("store")["footfall"].transform("mean")
    return df
"""

try:
    tools.transform_features(session, NAIVE)
except SandboxError as e:
    print(str(e))

The refusal is not pedantry. Only the *code* is stored (`0010`), so it is re-run
on whatever rows arrive at predict time — and a `groupby` over one row returns
that row's own value:

In [ ]:
one = df.iloc[[3]]
print("in the full frame:", df.groupby("store")["footfall"].transform("mean").iloc[3].round(2))
print("on its own:       ", one.groupby("store")["footfall"].transform("mean").iloc[0].round(2))

## 3. The encoder, by hand

`GroupStatsEncoder` is `FrequencyEncoder` with `value_counts` swapped for a stored
`groupby`. Fit computes the number per group; transform looks it up. Nothing is
recomputed, ever.

In [ ]:
from tabular_agent.encoders import GroupStatsEncoder

enc = GroupStatsEncoder(by=["store"], value="footfall", stat="mean").fit(df)
print("column:", enc.output_column)
print("groups learned:", len(enc.mapping_))
print("fallback for an unseen group:", round(enc.default_, 2))

The property the naive version fails, and the whole justification for this being a
tool rather than free code:

In [ ]:
whole = enc.transform(df)[enc.output_column].iloc[3]
alone = enc.transform(df.iloc[[3]])[enc.output_column].iloc[0]
print(f"row 3 in the full frame: {whole:.4f}")
print(f"row 3 on its own:        {alone:.4f}")
print("identical:", whole == alone)

And here is the consequence worth having. The lookup reads the **group key**, not
the column being summarised — so `footfall` does not have to exist at predict time
at all. A quantity you can measure in history but not on the day you predict
becomes usable *as its group's history*, never as itself.

In [ ]:
serving_row = pd.DataFrame({"store": ["s03"], "day": [999], "is_open": [1]})
enc.transform(serving_row)

In [ ]:
# A group never seen must not raise and must not be NaN. `mean` falls back to the
# population value; the extensive stats (`sum`, `count`, `nunique`) fall back to
# zero, because "we saw none of it" is the honest answer for those.
enc.transform(pd.DataFrame({"store": ["a-store-that-never-existed"]}))

## 4. Why it is always out-of-fold

The first version of this design said a per-group statistic of a *non-target*
column needs no out-of-fold discipline at all. That would have shipped a leak.

`footfall` correlates 0.99 with `sales` here. A per-store mean of `footfall` is
therefore very nearly a per-store mean of the target — and computing it including
the row's own contribution is target encoding with the guard taken off. The tool
cannot inspect the correlation to decide, so it always holds the row out.

In [ ]:
from tabular_agent.cv import make_folds
from tabular_agent.encoders import fit_group_stats

folds = make_folds(df["sales"], "regression", n_splits=4, seed=42)

oof, final = fit_group_stats(df, ["store"], "footfall", "mean", folds=folds)
in_fold = final.transform(df)[final.output_column]

pd.DataFrame({"out_of_fold": oof[final.output_column], "fitted_on_all": in_fold}).head()

At sixty rows per store the two barely differ — **and that is exactly the trap.**
A row moves its own group's mean by a sixtieth, so the leak is invisible and the
in-fold version looks fine. Group size is a property of the data, not of the tool.
Here is the same operation where each group has two rows:

In [ ]:
tiny = pd.DataFrame({
    "g": ["a", "a", "b", "b", "c", "c"],
    "v": [0.0, 100.0, 0.0, 100.0, 0.0, 100.0],
})
tiny_folds = [(np.array([1, 3, 5]), np.array([0, 2, 4])),
              (np.array([0, 2, 4]), np.array([1, 3, 5]))]

t_oof, t_final = fit_group_stats(tiny, ["g"], "v", "mean", folds=tiny_folds)
pd.DataFrame({
    "v": tiny["v"],
    "out_of_fold": t_oof[t_final.output_column],
    "fitted_on_all": t_final.transform(tiny)[t_final.output_column],
})

Read the middle column: a row holding `0.0` receives `100.0`, its partner's value,
because its own contribution was held out. The right-hand column — what a
prediction gets, where no row of the request is in the training data at all — is
`50.0` for everybody. Same arrangement as out-of-fold target encoding, same
reason (`docs/oof_vs_final_encoder.md`).

## 5. Through the harness

The tool writes a table plus a lineage sidecar, exactly like `fit_encoding`.

In [ ]:
stats = tools.fit_group_stats(session, by=["store"], value="footfall", stat="mean")
print(json.dumps({k: stats[k] for k in
                  ("table", "new_columns", "n_groups", "n_rows_without_value")}, indent=2))
print()
print(stats["note"])

The table still holds `footfall` itself, so training on it is refused. That is
`0030` doing its job and it is *not* a contradiction: the statistic is servable,
the raw column is not.

In [ ]:
refused = tools.train_tabular_model(session, table=stats["table"])
print(refused["trained"], "|", refused["reason"])

In [ ]:
DROP_FOOTFALL = """
def transform_features(df):
    return df.drop(columns=["footfall"], errors="ignore")
"""
usable = tools.transform_features(session, DROP_FOOTFALL, table=stats["table"])
exp = tools.train_tabular_model(session, table=usable["table"])
print("val_score:", round(exp["val_score"], 2), "| features:", exp["n_features"])
exp["columns_by_table"]

## 6. Dropping rows

`transform_features` cannot do it — it must preserve the row index — and the error
now says where to go instead.

In [ ]:
try:
    tools.transform_features(session, """
def transform_features(df):
    return df[df["is_open"] == 1]
""")
except SandboxError as e:
    print(str(e))

`filter_rows` takes a `keep_rows(df)` returning a **boolean mask**, not a filtered
frame. A frame could also reorder, rename or invent rows and each would have to be
checked for; a boolean Series indexed like the input can only select.

Two things it is allowed to do that `transform_features` is not, and both follow
from the same fact — the code is evaluated once and never replayed:

- it may read the **target** (outlier removal is ordinary modelling, and `0009`'s
  reason for hiding the target does not apply to code that is never re-run);
- it may be **batch-dependent** (`x > x.mean()` is fine; there is no second run
  for it to disagree with).

In [ ]:
KEEP_OPEN = """
def keep_rows(df):
    return df["is_open"] == 1
"""
filtered = tools.filter_rows(session, KEEP_OPEN, table=usable["table"])
print(json.dumps({k: filtered[k] for k in
                  ("table", "n_rows_before", "n_rows", "n_dropped", "fraction_dropped")}, indent=2))
print()
print(filtered["note"])

## 7. The load-bearing part

The filter is in the lineage, flagged `train_only`. Everything else in that list is
an instruction for predict time; this one is history.

In [ ]:
from tabular_agent.harness.tools import _load_lineage
from tabular_agent.io import load_table

lineage = _load_lineage(session.resolve_table(filtered["table"]), "regression")
for step in lineage.to_dict()["steps"]:
    print(f'{step["type"]:10s} train_only: {step.get("train_only", False)}')

Here is what would break if it were not. `Pipeline.transform` has exactly two
callers: `_score_on_val`, which replays a table's lineage over the validation half
and lines the predictions up against held-back targets, and `predict_bundle`,
which owes an answer for every row it is given. A filter running in the first
misaligns the score *and* grades a filtered experiment on an easier exam than an
unfiltered one. Every validation row has to survive the replay:

In [ ]:
val = load_table(session.val_path)
replayed = lineage.transform(val.drop(columns=["sales"]))
print("validation rows in: ", len(val))
print("validation rows out:", len(replayed))

In [ ]:
exp_filtered = tools.train_tabular_model(session, table=filtered["table"])
print(f"unfiltered  val {exp['val_score']:8.2f}   cv {exp['cv_score']:8.2f}")
print(f"filtered    val {exp_filtered['val_score']:8.2f}   cv {exp_filtered['cv_score']:8.2f}")

### Read that table before moving on — the filter made things much worse

This is the honest result and it is more useful than a win would have been.

**`cv_score` improved and `val_score` collapsed.** Both are correct. The filtered
model's cross-validation is computed over open days only — an easier population,
so a better number that is *not comparable* to the row above it. `val_score` is
measured on the whole validation half, including the closed days the model never
learned to predict, so it pays for them in full.

That is the arrangement working as designed: the exam did not move (`0029`), only
the studying did. Two lessons:

1. **A filter is graded on everything, including what it dropped.** Here the metric
   is plain `rmse` over all rows. Dropping the zero-sales days from the fit does
   not drop them from the score.
2. **`is_open` is a feature.** The unfiltered model can simply learn "shut → 0",
   which is a thing trees are extremely good at. When the regime is *expressible*,
   removing its rows takes information away and gives nothing back.

So when does the tool pay? When at least one of those is false — a metric that
genuinely does not score the dropped rows, or a regime the features cannot express.
Both are real; neither is this demo. **We do not manufacture the win.**

Check the diagnosis rather than trusting it, by scoring both models on the open
validation rows only:

In [ ]:
import joblib
from tabular_agent.metrics import compute_metric

open_val = (val["is_open"] == 1).to_numpy()
for name, e in [("unfiltered", exp), ("filtered", exp_filtered)]:
    art = os.path.join(session.run_dir, e["artifact_path"])
    meta = json.load(open(os.path.join(art, "meta.json")))
    model = joblib.load(os.path.join(art, "final_model.joblib"))
    feats = _load_lineage(session.resolve_table(e["source_table"]), "regression").transform(
        val.drop(columns=["sales"]))
    p = model.predict(feats[meta["feature_names"]])
    print(f"{name:11s} all rows {compute_metric('rmse', val['sales'], p):8.1f}"
          f"   open rows only {compute_metric('rmse', val['sales'][open_val], p[open_val]):8.1f}")

Even on the population it was trained for, the filtered model is not ahead. The
closed days were never costing it anything, because it could see `is_open`.

## 8. Serving

The last thing to check is the one that would be catastrophic and silent: a bundle
built from a filtered table still has to answer for every row, including rows from
the regime it never trained on.

In [ ]:
from tabular_agent.pipeline import load_bundle, predict_bundle

final_run = tools.finalize_run(session)
bundle = load_bundle(os.path.join(session.run_dir, final_run["bundle_dir"]))

request = val.drop(columns=["sales", "footfall"])   # no target, no footfall
preds = predict_bundle(bundle, request)
print("rows asked:", len(request), "| predictions:", len(preds),
      "| any missing:", bool(np.isnan(preds).any()))
print("of which closed days:", int((request["is_open"] == 0).sum()))

## 9. What the run remembers

A filter adds no columns, which would make it look like a wasted call in the
journal. `n_dropped` is why it does not.

In [ ]:
READ_BACK = """
def explore(tables):
    return tables[["id", "method", "n_columns", "n_dropped"]].to_dict("records")
"""
tools.explore_journal(session, READ_BACK)["result"]

## 10. Handing it to an agent

Everything above was driven by hand. The only new thing here is that nobody
mentions either tool — the question this section exists to answer is whether the
docstrings alone are enough for an agent to reach for them, which is a behavioural
result and the reason `scripts/run_rossmann.py` reports tool usage.

**This cell needs an API key (`config.env`) and costs money. It is not run when
the notebook ships.**

In [ ]:
from tabular_agent.harness.agent import Deps, build_agent, gateway_model

live = RunSession(
    os.path.join(WORK, "live"), data_path,
    TaskSpec(target="sales", task_type="regression", metric="rmse", n_splits=4),
    serving_columns=["store", "day", "is_open"],
)
agent = build_agent(gateway_model("tensorx/deepseek-v4-flash-0731"))
result = agent.run_sync(
    "Build the best model you can for this dataset of daily store sales. "
    "The target is sales.",
    deps=Deps(session=live),
)
print(result.output)

In [ ]:
# Which of the two new tools it reached for, and whether anything was trained on them.
from tabular_agent.harness import journal

j = journal.load(live.run_dir)
for t in j.get("tables") or []:
    print(t["id"], t["method"], t.get("new_columns"), t.get("n_dropped"))

## What to take away

- **`0011` was never wrong.** A per-group aggregate written as free code *is*
  batch-dependent and *should* be refused. What was missing was a place to put the
  operation where the value gets stored — and once it is stored, the objection
  disappears.
- **The out-of-fold path is not optional, and the reason is not the target
  column.** It is that the tool cannot tell whether the value column is a proxy for
  it. Large groups hide the leak rather than removing it.
- **A filter is the first pipeline step that is history rather than instruction.**
  Everything else in a lineage is replayed; this one must not be, or scoring
  misaligns and serving refuses rows.
- **The demo's filter lost, and the write-up says so.** `cv_score` improving while
  `val_score` collapses is not a paradox — it is two numbers measured on two
  populations, which is precisely what the note on the tool's result warns about.